# Feature Extraction — ResNet50
Pass each resized image through ResNet50 (final layer removed) to get a 2048-dim feature vector per image.

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torchvision.models as models
import torchvision.transforms as transforms

PROC_ROOT  = r'..\data\processed'
IMG_DIR    = os.path.join(PROC_ROOT, 'images')
ANN_CSV    = os.path.join(PROC_ROOT, 'annotations.csv')
OUT_PATH   = os.path.join(PROC_ROOT, 'image_features.npy')
IDS_PATH   = os.path.join(PROC_ROOT, 'image_ids.npy')

SPLITS = ['train2017', 'val2017', 'test2017']

# ImageNet normalization — same values ResNet50 was trained with
transform = transforms.Compose([
    transforms.ToTensor(),                          # PIL → tensor, scales to [0, 1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## 1. Load ResNet50 — Remove Final Layer

In [2]:
# Load pretrained ResNet50
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Remove the final fully-connected layer (1000-class classifier)
# nn.Sequential(*list(...)[:-1]) keeps everything except the last layer
import torch.nn as nn
model = nn.Sequential(*list(resnet.children())[:-1])
model = model.to(device)
model.eval()   # freeze — no training, no dropout

print('ResNet50 loaded. Final layer removed.')
print(f'Output shape per image: (2048, 1, 1) → flattened to (2048,)')

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\armaa/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:03<00:00, 32.2MB/s]


ResNet50 loaded. Final layer removed.
Output shape per image: (2048, 1, 1) → flattened to (2048,)


## 2. Verify Output Shape on One Image

In [3]:
sample_path = os.path.join(IMG_DIR, 'train2017', '000001.jpg')
sample_img  = Image.open(sample_path).convert('RGB')
sample_tensor = transform(sample_img).unsqueeze(0).to(device)  # add batch dim → (1, 3, 224, 224)

with torch.no_grad():
    out = model(sample_tensor)

print(f'Input tensor shape  : {sample_tensor.shape}')   # (1, 3, 224, 224)
print(f'Raw output shape    : {out.shape}')              # (1, 2048, 1, 1)
print(f'Flattened shape     : {out.squeeze().shape}')    # (2048,)
print(f'Sample values       : {out.squeeze()[:5].cpu().numpy()}')

Input tensor shape  : torch.Size([1, 3, 224, 224])
Raw output shape    : torch.Size([1, 2048, 1, 1])
Flattened shape     : torch.Size([2048])
Sample values       : [0.53753823 0.52121115 0.13244423 0.7900434  0.0254166 ]


## 3. Extract Features for All Images

In [4]:
# Load annotation CSV to get the ordered list of image_ids and filenames
df = pd.read_csv(ANN_CSV)
print(f'Total images to process: {len(df)}')
df.head(3)

Total images to process: 4000


,image_id,file_name,split,damage_count,total_damage_area,dominant_damage,has_dent,has_scratch,has_crack,has_glass_shatter,has_lamp_broken,has_tire_flat
0,1,000001.jpg,train2017,2,312326.0,tire flat,0,1,0,0,0,1
1,2,000002.jpg,train2017,1,114126.0,tire flat,0,0,0,0,0,1
2,3,000003.jpg,train2017,1,299792.0,tire flat,0,0,0,0,0,1


In [5]:
features = []
image_ids = []

with torch.no_grad():
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting features'):
        img_path = os.path.join(IMG_DIR, row['split'], row['file_name'])

        img    = Image.open(img_path).convert('RGB')
        tensor = transform(img).unsqueeze(0).to(device)  # (1, 3, 224, 224)

        feat = model(tensor)           # (1, 2048, 1, 1)
        feat = feat.squeeze().cpu().numpy()  # (2048,)

        features.append(feat)
        image_ids.append(row['image_id'])

features   = np.array(features)    # (4000, 2048)
image_ids  = np.array(image_ids)   # (4000,)

print(f'\nFeature matrix shape : {features.shape}')
print(f'Image IDs shape      : {image_ids.shape}')

Extracting features: 100%|██████████| 4000/4000 [08:15<00:00,  8.07it/s]



Feature matrix shape : (4000, 2048)
Image IDs shape      : (4000,)


## 4. Save to Disk

In [6]:
np.save(OUT_PATH, features)
np.save(IDS_PATH, image_ids)

size_mb = os.path.getsize(OUT_PATH) / 1024 / 1024
print(f'Saved image_features.npy  →  {features.shape}  ({size_mb:.1f} MB)')
print(f'Saved image_ids.npy       →  {image_ids.shape}')

Saved image_features.npy  →  (4000, 2048)  (31.3 MB)
Saved image_ids.npy       →  (4000,)


## 5. Sanity Check

In [7]:
# Reload and verify
loaded = np.load(OUT_PATH)
ids    = np.load(IDS_PATH)

print(f'Loaded features shape : {loaded.shape}')       # (4000, 2048)
print(f'Loaded ids shape      : {ids.shape}')           # (4000,)
print(f'Feature value range   : [{loaded.min():.3f}, {loaded.max():.3f}]')
print(f'Any NaNs              : {np.isnan(loaded).any()}')
print(f'First image_id        : {ids[0]}  → first 5 features: {loaded[0][:5]}')

Loaded features shape : (4000, 2048)
Loaded ids shape      : (4000,)
Feature value range   : [0.000, 9.170]
Any NaNs              : False
First image_id        : 1  → first 5 features: [0.53753823 0.52121115 0.13244423 0.7900434  0.0254166 ]
